# 🛒 Product Recommendation System
## Using PySpark MLlib (ALS) + Neo4j Graph Database

---

**Author:** osman  
**Project:** Product-Recommendation-ML-PySpark-Neo4j  

---

### 📌 Project Overview
This notebook demonstrates a **hybrid product recommendation system** that combines:
- **PySpark MLlib ALS (Alternating Least Squares)** — Collaborative Filtering for user-product recommendations
- **Neo4j Graph Database** — Graph-based relationship traversal for content-based and graph recommendations

### 🏗️ Architecture
```
Raw Data (CSV/Parquet)
       │
       ▼
PySpark Data Processing
       │
       ├──► ALS Model (Collaborative Filtering)
       │           │
       │           ▼
       │    Top-N Recommendations
       │
       └──► Neo4j Graph Store
                   │
                   ▼
            Graph-based Recommendations
                   │
                   ▼
         Final Hybrid Recommendations
```

### 📦 Tech Stack
| Component | Technology |
|-----------|------------|
| Data Processing | Apache PySpark 3.x |
| ML Algorithm | PySpark MLlib — ALS |
| Graph Database | Neo4j 5.x |
| Graph Driver | neo4j-python-driver |
| Visualization | Matplotlib, Seaborn |
| Language | Python 3.9+ |

## 📋 Table of Contents
1. [Environment Setup & Imports](#1-environment-setup)
2. [Data Generation / Loading](#2-data-loading)
3. [Exploratory Data Analysis (EDA)](#3-eda)
4. [PySpark Data Preprocessing](#4-preprocessing)
5. [ALS Collaborative Filtering Model](#5-als-model)
6. [Model Evaluation](#6-evaluation)
7. [Neo4j — Graph Setup & Data Ingestion](#7-neo4j-setup)
8. [Graph-Based Recommendations (Cypher)](#8-graph-recommendations)
9. [Hybrid Recommendations](#9-hybrid)
10. [Results & Visualization](#10-results)
11. [Conclusion](#11-conclusion)

---
## 1. Environment Setup & Imports
<a id='1-environment-setup'></a>

In [ ]:
# Install required packages (run once)
# !pip install pyspark neo4j pandas numpy matplotlib seaborn faker

print("✅ Dependencies ready")

In [ ]:
# ─────────────────────────────────────────────────
#  Standard Libraries
# ─────────────────────────────────────────────────
import os
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ─────────────────────────────────────────────────
#  PySpark
# ─────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, FloatType, StringType, TimestampType
)
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# ─────────────────────────────────────────────────
#  Neo4j
# ─────────────────────────────────────────────────
from neo4j import GraphDatabase

print("✅ All imports successful")

In [ ]:
# ─────────────────────────────────────────────────
#  Initialize Spark Session
# ─────────────────────────────────────────────────
spark = SparkSession.builder \
    .appName("Product-Recommendation-PySpark-Neo4j") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print(f"✅ Spark version: {spark.version}")
print(f"✅ App Name: {spark.sparkContext.appName}")